# First assignment - short description

Student: Magdalena Czapiewska

This notebook contains the solution to the first assignment (clustering letters).
It is divided into 6 sections:

1. Creating execution environment
2. Custom options adjustment
3. Data preparation
4. Features extraction
5. Clustering algorithm
6. Evaluation (this section does not execute, it just presents the experiments I have made to choose the best custom options)

The pipeline begins with resizing images so each of them is of the same size. Then features are extracted using DAISY method (Histogram of oriented gradients method was also considered, but not chosen). Feature vectors are clustered using Agglomerative clustering with linkage method `ward` and `euclidean` distance metric.

In each section I describe in detail the algorithms used. Custom options can be changed in the `Custom options adjustment` section (paths to input and output files can be changed here if they differ from the assignment description).

# First assignment - how to run

#### Step 1. Environment setup
I have run this notebook using jupyter-notebook. To successfully run the notebook, please run the code in the `Creating execution environment` section. Then a new kernel called `magdalena_czapiewska_sus_1_env` will be added to jupyter-notebook.

#### Step 2. Switching the kernel
Please select from the menu `Kernel -> Change Kernel -> magdalena_czapiewska_sus_1_env`.

#### Step 3. Execution
Please run all cells starting from the first cell in the `Custom options adjustment` section (`Run -> Run Selected Cell and All Below`). If it does not work, please execute each cell independently (time of execution is less than 5 minutes for the whole notebook after kernel choice).

#### Step 4. In case of failure of kernel registration / activation
If you encounter any problems with activating a kernel, please use an alternative option. I provided a python script magdalena_czapiewska_zal1.py that has the same functionality as this notebook (custom options are constants at the beginning of the script). To execute it, please create conda virtual environment:

conda create -y -n magdalena_czapiewska_sus_1_conda -c conda-forge python=3.10 "numpy<2" "opencv<4.10" matplotlib scikit-learn scikit-image

conda activate magdalena_czapiewska_sus_1_conda

python magdalena_czapiewska_zal1.py

#### Time of execution for 7620 images

##### For python script in conda environment:

real	1m17,799s
user	1m13,336s
sys	0m1,381s

##### For ipynb file:

Imports:
CPU times: user 1.41 s, sys: 203 ms, total: 1.61 s
Wall time: 1.16 s

Loading images:
CPU times: user 580 ms, sys: 186 ms, total: 766 ms
Wall time: 774 ms

DAISY feature extraction:
CPU times: user 1min 11s, sys: 11.2 ms, total: 1min 11s
Wall time: 1min 11s

Clustering:
CPU times: user 19.3 s, sys: 158 ms, total: 19.5 s
Wall time: 19.5 s

---------------------------------------------------------------------------------------------------------------------

## 1. Creating execution environment

After running the cell below, please select from the menu `Kernel -> Change Kernel -> magdalena_czapiewska_sus_1_env` and run all cells starting from the first cell in the `Custom options adjustment` section.

In [10]:
import sys
import os

executable = sys.executable

!{executable} -m venv magdalena_czapiewska_sus_1_env

venv_pip = "./magdalena_czapiewska_sus_1_env/bin/pip"
venv_python = "./magdalena_czapiewska_sus_1_env/bin/python"

!{venv_pip} install ipykernel
!{venv_pip} install "numpy<2" "opencv-python<4.10" matplotlib scikit-learn scikit-image
!{venv_python} -m ipykernel install --user --name=magdalena_czapiewska_sus_1_env --display-name "magdalena_czapiewska_sus_1_env"

print("\n--- Environment created successfully! Please switch the kernel now. ---")

  Using cached ipykernel-7.2.0-py3-none-any.whl (118 kB)
  Using cached debugpy-1.8.20-cp310-cp310-manylinux_2_34_x86_64.whl (3.1 MB)
  Using cached packaging-26.2-py3-none-any.whl (100 kB)
  Using cached pyzmq-27.1.0-cp310-cp310-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (854 kB)
  Using cached ipython-8.39.0-py3-none-any.whl (831 kB)
  Using cached traitlets-5.15.0-py3-none-any.whl (85 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl (5.2 kB)
  Using cached comm-0.2.3-py3-none-any.whl (7.3 kB)
  Using cached matplotlib_inline-0.2.2-py3-none-any.whl (9.5 kB)
  Using cached jupyter_client-8.8.0-py3-none-any.whl (107 kB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl (155 kB)
  Using cached tornado-6.5.5-cp39-abi3-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (447 kB)
  Using cached jupyter_core-5.9.1-py3-none-any.whl (29 kB)
  Using cached jedi-0.20.0-py2.py3-none-any.whl (4.9 MB)
  Using cached p

-----------------------------------------------------------------------------------------------------------------

## 2. Custom options adjustment

**SIZE** - each image will be resized to image of size SIZE x SIZE

**INPUT_FILE** - Input to the program. A text file that contains in each line the name (and possibly the path) of the image file

**OUTPUT_HTML** - Output of the program. File in html format that displays images belonging to individual clusters

**OUTPUT_TXT** - Output of the program. A file containing in each line a space-separated  list of files containing images belonging to the given cluster. The ends of the lines in the unix convention. The output file should contain only the files' names of the images that belong to the same cluster which are space separated & without any additional text related to the directory

**THRESHOLD** - Threshold used for forming clusters from the hierarchical tree

**GROUND_TRUTH** - A file with ground truth clusters for images in training_samples directory. Used for evaluation of the method. Described in greater detail in the `Evaluation` section.

**FEATURES_EXTRACTION_ALGORITHM** - Features extraction algorithm (possible values: hog, daisy)

**LINKAGE_METHOD** - Linkage method used in agglomerative clustering (possible values: ward, single, average, complete)

In [1]:
SIZE = 64
INPUT_FILE = "input.txt"
OUTPUT_HTML = "output_clusters_images.html"
OUTPUT_TXT = "output_clusters.txt"
THRESHOLD = 1.4
GROUND_TRUTH = "ground_truth_clusters.txt"
FEATURES_EXTRACTION_ALGORITHM = "daisy" # possible values: hog, daisy
LINKAGE_METHOD = "ward" # possible values: ward, single, average, complete

In [2]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import hog, daisy
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, rand_score

## 3. Data preparation

The pipeline begins with resizing images. There are 3 goals:
1. Resize each image so it is of size $SIZE \times SIZE$ 
2. Preserve the proportions of the original letter (so that '.' and 'l' differ after resizing to square image)
3. Minimizing the effect of shifting the letters on the pictures (a letter should be in the center of the resulting image)

Preprocessing steps:
1. Binarization: Images are loaded in grayscale (0–255). I invert the values so that the character becomes foreground (white) and the background becomes black. Otsu’s method is applied to automatically determine the optimal threshold for separating the character from the background, creating a binary mask.
2. Cropping (bounding box): Using the binary mask, I locate all non-zero pixels to find the character's coordinates. I calculate the smallest possible bounding box and crop the original grayscale image to these dimensions, effectively removing all empty margins.
3. Proportional resizing: I calculate a scaling factor based on the larger dimension (width or height) of the cropped character relative to the target SIZE. This ensures the character fits within the target boundaries without stretching or distortion.
4. Centering and padding: The resized character is placed in the center of a new $SIZE \times SIZE$ white canvas. This step provides the "shift invariance" required by the task.
5. Normalization: Finally, pixel values are normalized to a floating-point range of $[0.0, 1.0]$.

In [3]:
def process_image(file_path, size, debug=False):
    """
    Loads, preprocesses, and normalizes a character image.
    
    Args:
        file_path (str): Path to the image file.
        size (int): Target size (width and height) of the output image.
        debug (bool): If True, displays the original and processed image with matrix values.
        
    Returns:
        numpy.ndarray: Normalized 2D image of shape (size, size) with float32 values, 
                       or None if loading fails.
    """
    # img is a 2-dimensional numpy.ndarray with entities of type uint8
    # each pixel is represented by one entity
    # values range from 0 (black) to 255 (white)
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Error: Could not load file {file_path}")
        return None

    # The goal of the following operations is to find a bounding box around the letter in the image
    # values are inversed so that the letter is white (255) and the backgound is black (0)
    # Otsu's algorithm is used for analysing histogram of pixel values and choosing a threshold value
    # that separates letter and background
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # coordinates of non-zero pixels (white, letter) are found 
    coords = cv2.findNonZero(binary)
    if coords is None:
        return np.ones((size, size), dtype=np.float32)

    # the smallest rectangle (x, y, width, height) that contains all non-zero pixels is calculated
    x, y, w, h = cv2.boundingRect(coords)
    # the original grayscale image is cropped to this rectangle to remove empty margins
    # x, y: coordinates of the top-left corner of the bounding box
    # w, h: width and height of the box
    # slicing in NumPy [y:y+h, x:x+w] extracts the region of interest
    # by selecting rows (vertical range) and columns (horizontal range)
    # cropped_original contains a letter (0, black, as original) and fragments of background (255, white, original)
    cropped_original = img[y:y+h, x:x+w]

    # To avoid distorting the character, its original aspect ratio is preserved.
    # The scaling factor is calculated based on the larger dimension (width or height),
    # ensuring the resized character fits within a SIZE x SIZE box
    scaling_factor = float(size) / max(h, w)
    new_w = int(w * scaling_factor)
    new_h = int(h * scaling_factor)

    # Fragment of the original image (center with a letter, without margins) is resized
    # so that the larger dimension reaches SIZE and proportions are preserved.
    # INTER_AREA algorithm overlays a grid of the target resolution onto the source image.
    # It calculates the values of a new pixel by looking at the overlap with the original pixels
    # and computing the weighted average of all pixels covered by the target pixel's area.
    resized = cv2.resize(cropped_original, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # canvas is white (255)
    canvas = np.full((size, size), 255, dtype=np.uint8)
    # The top-left corner coordinates (start_x, start_y) are calculated
    # to place the character in the center of the canvas.
    start_x = (size - new_w) // 2
    start_y = (size - new_h) // 2
    canvas[start_y:start_y+new_h, start_x:start_x+new_w] = resized

    # values are normalized to 0.0 (black) - 1.0 (white) range (float32)
    final_img_norm = canvas.astype(np.float32) / 255.0

    if debug:
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(img, cmap='gray')
        ax[0].set_title("Original")
        
        ax[1].imshow(final_img_norm, cmap='gray', vmin=0, vmax=1)
        ax[1].set_title(f"Normalized {size}x{size} (0-1)")
        plt.show()
        
        start_idx = size // 2 - 5
        end_idx = start_idx + 10
        
        print(f"\n### MATRIX VALUES (10x10 center fragment from index {start_idx} to {end_idx}) ###")
        print(final_img_norm[start_idx:end_idx, start_idx:end_idx])

    return final_img_norm

In [4]:
def load_image_paths(file_path):
    """
    Reads image file paths from a text file.
    
    Args:
        file_path (str): Path to the input file.
        
    Returns:
        list: A list of cleaned strings, each representing a path to an image.
    """
    image_paths = []

    if not os.path.exists(file_path):
        print(f"Error: File {file_path} not found.")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # .strip() removes leading/trailing whitespaces and newline characters (\n)
            path = line.strip()
            if path:
                image_paths.append(path)
                
    return image_paths

def generate_reports(clusters_dict, output_html, output_txt):
    """
    Generates HTML and TXT reports based on clustering results.
    
    Args:
        clusters_dict (dict): Dictionary where keys are cluster IDs and values are lists of image paths.
        output_html (str): Filename for the HTML output.
        output_txt (str): Filename for the TXT output.
    """
    active_cluster_ids = sorted([cid for cid, paths in clusters_dict.items() if paths])

    with open(output_html, "w", encoding="utf-8") as f:
        f.write("<html><body>\n")
        
        for i, cid in enumerate(active_cluster_ids):
            for path in clusters_dict[cid]:
                filename = os.path.basename(path)
                f.write(f'<img src="{path}" title="{filename}" style="margin:2px; max-height:50px;">\n')
            
            if i < len(active_cluster_ids) - 1:
                f.write("<HR>\n")
                
        f.write("</body></html>\n")

    with open(output_txt, "w", encoding="utf-8", newline="\n") as f:
        for cid in active_cluster_ids:
            filenames = [os.path.basename(p) for p in clusters_dict[cid]]
            f.write(" ".join(filenames) + "\n")

def process_and_vectorize_images(paths, size):
    """
    Iterates through a list of image paths, processes each image, and flattens it.
    
    Args:
        paths (list): List of image file paths.
        size (int): The target size for resizing images.
        
    Returns:
        tuple: (X, valid_paths) where:
            - X (numpy.ndarray): Feature matrix of shape (n_samples, size*size).
            - valid_paths (list): List of paths corresponding to the rows in X.
    """
    processed_images = []
    valid_paths = []

    for p in paths:
        proc_img = process_image(p, size, debug=False)
        
        if proc_img is not None:
            processed_images.append(proc_img.flatten())
            valid_paths.append(p)

    X = np.array(processed_images)
    return X, valid_paths

## 4. Features extraction

I want to extract features from images to better describe shapes. I don't want the clustering to be affected by differences of single pixels. I try out 2 features extraction algorithms: Histogram of Oriented Gradients and DAISY. I choose DAISY method (details in the `Evaluation` section).

**Histogram of Oriented Gradients** description (https://www.geeksforgeeks.org/computer-vision/histogram-of-oriented-gradients/):

1. Image is converted to grayscale and resized

2. The algorithm calculates the horizontal ($G_x$) and vertical ($G_y$) gradients for every single pixel (measuring how brightness change horizontally and vertically). This is usually done using a Sobel filter. Magnitude ($\sqrt{G_x^2 + G_y^2}$) represents the "strength" or intensity of the edge. Direction ($arctan2(G_x, G_y)$) represents the "orientation" (angle) of the edge.

3. The image is divided into small square regions called cells (typically 8x8 pixels, in my project 4x4 pixels). For each cell, a histogram of gradients is created. Usually, there are 9 bins representing angles from 0 to 180 degrees (in 20-degree increments). Every pixel in the cell "votes" for a bin based on its gradient direction, and the "weight" of that vote is determined by its gradient magnitude.

4. Gradients are sensitive to lighting and contrast. To fix this, HOG groups cells into larger, overlapping blocks (in my project 2x2 cells). The algorithm calculates a normalization factor for the entire block.
5. All the normalized histograms from all the blocks are "flattened" and concatenated into one long, continuous array of numbers.

**DAISY method** description ("A Fast Local Descriptor For Dense Matching", Tola, Lepetit, Fua):

1. The process begins by calculating gradient maps of the image for several orientations (usually 8). These maps represent the strength of gradients in specific directions for every pixel.

2. Each orientation map is convolved with Gaussian kernels of varying standard deviations (it is called Gaussian smoothing). It ensures that the descriptor captures information at different spatial resolutions and becomes indifferent to small local shifts and noise.

3. Unlike the square grids used in SIFT, DAISY uses a circular (flower-like) sampling layout. It samples values from the convolved orientation maps at points located on concentric circles around a central pixel.

4. The values sampled from the different orientation maps at each grid point are concatenated into a single feature vector. This vector describes the local structure of the image around the pixel.

5. The resulting vector is normalized to make the descriptor invariant to global illumination changes, which is vital for maintaining consistency across different images.

In [5]:
def extract_hog_features(X_images, size, orientations=8, pixels_per_cell=(4, 4), cells_per_block=(2, 2)):
    """
    Extracts Histogram of Oriented Gradients (HOG) features from a set of images.
    
    Args:
        X_images (numpy.ndarray): Matrix of flattened images.
        size (int): The original side length of the square images (size x size).
        orientations (int): Number of orientation bins.
        pixels_per_cell (tuple): Size (in pixels) of a cell.
        cells_per_block (tuple): Number of cells in each block.
        
    Returns:
        numpy.ndarray: A 2D array where each row is a HOG feature vector for an image.
    """
    hog_features = []
    
    for img_flat in X_images:
        img = img_flat.reshape(size, size)
        
        features = hog(img, 
                       orientations=orientations, 
                       pixels_per_cell=pixels_per_cell,
                       cells_per_block=cells_per_block, 
                       visualize=False)
        
        hog_features.append(features)
        
    return np.array(hog_features)

def extract_daisy_features(X_images, size, step=16, radius=15, rings=2, histograms=6, orientations=8):
    """
    Extracts DAISY features from a set of images.
    
    Args:
        X_images (numpy.ndarray): Matrix of flattened images.
        size (int): The original side length of the square images (size x size).
        step (int): Distance between descriptor sampling points.
        radius (int): Radius of the outermost ring.
        rings (int): Number of rings in the descriptor grid.
        histograms (int): Number of histograms sampled per ring.
        orientations (int): Number of orientations in each histogram.
        
    Returns:
        numpy.ndarray: A 2D array of flattened DAISY descriptors for each image.
    """
    daisy_features = []
    
    for img_flat in X_images:
        img = img_flat.reshape(size, size)

        features = daisy(img, 
                         step=step, 
                         radius=radius, 
                         rings=rings, 
                         histograms=histograms, 
                         orientations=orientations, 
                         visualize=False)

        daisy_features.append(features.flatten())
        
    return np.array(daisy_features)

## 5. Clustering algorithm

I use Agglomerative Clustering method as it does not require predefining the number of clusters. I choose the threshold value that maximizes Rand Index and Adjusted Rand Index value on the provided training_samples dataset.

Agglomerative Clustering description (https://www.geeksforgeeks.org/machine-learning/agglomerative-clustering/):

1. At the beginning each data point is treated as its own cluster
2. Pairwise distances between clusters are computed and stored in the distance matrix
3. Two clusters that are closest based on the chosen linkage method are identified. They are combined into a single new cluster
4. Distances between newly formed cluster and all remaning clusters are recalculated
5. Merging clusters is continued
6. The process is stopped when a predefined number of clusters or a distance threshold is reached (in our case it is distance threshold)

To define the distance between 2 clusters I use the Euclidean distance (https://www.geeksforgeeks.org/maths/euclidean-distance/).

Euclidean distance between 2 points $(x_{1,1}, x_{1,2}, ..., x_{1,n})$ and $(x_{2,1}, x_{2,2}, ..., x_{2,n})$ in an n-dimensional space is given by the formula:

$$d = \sqrt{\sum_{i=1}^n(x_{2,i}-x_{1,i})^2}$$

Among 4 different linkage methods (single, complete, average, Ward's) I try out average and Ward's and choose Ward's as it provides better and more stable results (small differences in the threshold does not affect the values of RI and ARI so much). The detalis are in `Evaluation` section.

Description of the linkage methods (https://www.geeksforgeeks.org/machine-learning/ml-types-of-linkages-in-clustering/): 

**Average linkage** returns the average distance between all pairs of points from two clusters.

$$L(R, S) = \frac{1}{n_R \times n_S}\sum_{i=1}^{n_R} \sum_{j=1}^{n_S} D(i, j),\;\;i \in R, j \in S$$


where

- $n_R$ and $n_S$ are the sizes of clusters $R$ and $S$


**Ward's linkage** calculates the distance between two clusters by looking at total spread or variance increase when the clusters are combined.

$$L(R, S) = \frac{n_R + n_S}{n_R \times n_S}\sum_{i=1}^{n_R} \sum_{j=1}^{n_S} D(i, j),\;\;i \in R, j \in S$$


where

- $n_R$ and $n_S$ are the sizes of clusters $R$ and $S$
- $D(i, j)$ is the distance between points $i \in R$ and $j \in S$

In [6]:
def perform_clustering(X_features, valid_paths, n_clusters=None, distance_threshold=0.5, metric='euclidean', linkage='ward'):
    """
    Performs hierarchical agglomerative clustering on the extracted features.
    
    Args:
        X_features (numpy.ndarray): Feature matrix (n_samples, n_features).
        valid_paths (list): List of image paths corresponding to the rows in X_features.
        n_clusters (int, optional): The number of clusters to find. Should be None if 
                                     distance_threshold is used.
        distance_threshold (float): The linkage distance threshold above which 
                                    clusters will not be merged.
        metric (str): Metric used to compute the linkage.
        linkage (str): Linkage criterion.
        
    Returns:
        dict: A dictionary where keys are cluster labels and values are lists of 
              image paths belonging to each cluster.
    """
    model = AgglomerativeClustering(
        n_clusters=n_clusters,
        distance_threshold=distance_threshold,
        metric=metric,
        linkage=linkage
    )

    labels = model.fit_predict(X_features)

    clusters_dict = {}
    for path, label in zip(valid_paths, labels):
        clusters_dict.setdefault(label, []).append(path)
    
    return clusters_dict

## Execution

In [7]:
all_paths = load_image_paths(INPUT_FILE)
X, valid_paths = process_and_vectorize_images(all_paths, SIZE)

print(f"Total paths read from {INPUT_FILE}: {len(all_paths)}")
print(f"Successfully processed images:    {len(valid_paths)}")

if X.size > 0:
    print(f"Matrix X shape (samples, features): {X.shape}")
    print(f"Feature vector size per image:      {X.shape[1]} (equal to {SIZE}x{SIZE})")
else:
    print("Warning: Matrix X is empty. Check your image paths or process_image function.")

Total paths read from input.txt: 7620
Successfully processed images:    7620
Matrix X shape (samples, features): (7620, 4096)
Feature vector size per image:      4096 (equal to 64x64)


In [8]:
if FEATURES_EXTRACTION_ALGORITHM.lower() == "daisy":
    print(f"Extracting features using DAISY algorithm...")
    X_features = extract_daisy_features(X, SIZE)
    
elif FEATURES_EXTRACTION_ALGORITHM.lower() == "hog":
    print(f"Extracting features using HOG algorithm...")
    X_features = extract_hog_features(X, SIZE)
    
else:
    print("Warning: Unknown algorithm selected. Using raw pixel values.")
    X_features = X

if X_features.size > 0:
    print("-" * 30)
    print(f"### FEATURE EXTRACTION SUMMARY ###")
    print(f"Algorithm used:           {FEATURES_EXTRACTION_ALGORITHM.upper()}")
    print(f"Feature Matrix shape:     {X_features.shape}")
    print(f"Number of samples:        {X_features.shape[0]}")
    print(f"Features per image:       {X_features.shape[1]}")
    print("-" * 30)
else:
    print(f"Warning: Feature matrix is empty! Check the {FEATURES_EXTRACTION_ALGORITHM} settings.")

Extracting features using DAISY algorithm...
------------------------------
### FEATURE EXTRACTION SUMMARY ###
Algorithm used:           DAISY
Feature Matrix shape:     (7620, 936)
Number of samples:        7620
Features per image:       936
------------------------------


In [9]:
clusters = perform_clustering(
    X_features, 
    valid_paths, 
    n_clusters=None, 
    distance_threshold=THRESHOLD, 
    metric='euclidean',
    linkage=LINKAGE_METHOD
)

generate_reports(clusters, OUTPUT_HTML, OUTPUT_TXT)

print("-" * 30)
print("### FINAL REPORT SUMMARY ###")
print(f"Features extraction algorithm used:    {FEATURES_EXTRACTION_ALGORITHM.upper()}")
print(f"Linkage method:    {LINKAGE_METHOD}")
print(f"Threshold:         {THRESHOLD}")
print(f"Clusters found:    {len(clusters)}")
print("-" * 30)
print(f"Results successfully saved to:")
print(f" - {OUTPUT_HTML}")
print(f" - {OUTPUT_TXT}")
print("-" * 30)

------------------------------
### FINAL REPORT SUMMARY ###
Features extraction algorithm used:    DAISY
Linkage method:    ward
Threshold:         1.4
Clusters found:    49
------------------------------
Results successfully saved to:
 - output_clusters_images.html
 - output_clusters.txt
------------------------------


## 6. Evaluation

I had 3 choices to make:
- feature extraction algorithm (DAISY or HOG)
- linkage method (average or Ward's)
- threshold

I evaluated each pair of **feature extraction algorithm** and **linkage method** for **thresholds reasonable for this pair**. The evaluation metrics used were Rand Index and Adjusted Rand Index. The results are shown in the tables below.

As ground truth clusters I used clusters that I created manually by modifying the results of my imperfect clustering. I provide the files **ground_truth_clusters.txt** (which I used for calculating evaluation metrics) and **clean_clusters.html** (which enables assessing ground truth clusters visually). It is essential to place **clean_clusters.html** file in the same folder where a folder **training_samples** exists to visualize the clusters.

The HOG algorithm (regardless of the linkage method) yields poor Adjusted Rand Index values. I observed that when using HOG, the clustering algorithm struggles to group all images of a single character into one dedicated cluster. While increasing the threshold should theoretically enable the merging of separate clusters representing the same character, it instead causes different characters to be merged into the same cluster, while images of the same character often remain split.

The DAISY algorithm enables reaching ARI equal to 0.8420 (for Ward's linkage with threshold 1.4) and ARI equal to 0.8544 (for average linkage with threshold 0.17). I choose Ward's linkage with threshold 1.4, as the results for average linkage are less stable (increasing the threshold to 0.25 results in ARI equal to 0.3234).

| Algorithm | Linkage | Threshold | Rand Index (RI) | Adjusted Rand Index (ARI) | Clusters (n) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| DAISY | Ward | 1.0 | 0.9736 | 0.6426 | 74 |
| DAISY | Ward | 1.1 | 0.9780 | 0.7201 | 64 |
| DAISY | Ward | 1.2 | 0.9797 | 0.7487 | 57 |
| DAISY | Ward | 1.3 | 0.9790 | 0.7519 | 54 |
| DAISY | Ward | 1.35 | 0.9790 | 0.7419 | 54 |
| **DAISY** | **Ward** | **1.4** | **0.9859** | **0.8420** | **49** |
| DAISY | Ward | 1.45 | 0.9846 | 0.8290 | 47 |
| DAISY | Ward | 1.5 | 0.9845 | 0.8286 | 46 |
| DAISY | Ward | 1.6 | 0.9845 | 0.8293 | 41 |
| DAISY | Ward | 1.7 | 0.9841 | 0.8251 | 39 |
| DAISY | Ward | 1.8 | 0.9841 | 0.8251 | 39 |
| DAISY | Ward | 1.9 | 0.9820 | 0.8071 | 37 |
| DAISY | Ward | 2.0 | 0.9786 | 0.7786 | 34 |


| Algorithm | Linkage | Threshold | Rand Index (RI) | Adjusted Rand Index (ARI) | Clusters (n) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| DAISY | Average | 0.1 | 0.9659 | 0.4376 | 938 |
| DAISY | Average | 0.13 | 0.9828 | 0.7815 | 571 |
| DAISY | Average | 0.15 | 0.9865 | 0.8361 | 442 |
| DAISY | Average | 0.16 | 0.9872 | 0.8472 | 384 |
| **DAISY** | **Average** | **0.17** | **0.9876** | **0.8544** | **339** |
| DAISY | Average | 0.18 | 0.9818 | 0.8000 | 292 |
| DAISY | Average | 0.2 | 0.9794 | 0.7809 | 228 |
| DAISY | Average | 0.25 | 0.8673 | 0.3234 | 125 |
| DAISY | Average | 0.3 | 0.5480 | 0.0685 | 70 |


| Algorithm | Linkage | Threshold | Rand Index (RI) | Adjusted Rand Index (ARI) | Clusters (n) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| HOG | Ward | 30.0 | 0.9611 | 0.3222 | 137 |
| HOG | Ward | 40.0 | 0.9628 | 0.3965 | 86 |
| HOG | Ward | 50.0 | 0.9650 | 0.4706 | 63 |
| HOG | Ward | 60.0 | 0.9662 | 0.5462 | 45 |
| HOG | Ward | 70.0 | 0.9654 | 0.5526 | 38 |
| HOG | Ward | 75.0 | 0.9634 | 0.5423 | 34 |
| **HOG** | **Ward** | **80.0** | **0.9624** | **0.5658** | **27** |
| HOG | Ward | 85.0 | 0.9613 | 0.5586 | 26 |
| HOG | Ward | 90.0 | 0.9593 | 0.5493 | 24 |
| HOG | Ward | 100.0 | 0.9534 | 0.5263 | 20 |
| HOG | Ward | 110.0 | 0.9518 | 0.5290 | 19 |
| HOG | Ward | 120.0 | 0.9365 | 0.4741 | 16 |


| Algorithm | Linkage | Threshold | Rand Index (RI) | Adjusted Rand Index (ARI) | Clusters (n) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| HOG | Average | 10.0 | 0.9595 | 0.2838 | 1519 |
| HOG | Average | 11.0 | 0.9640 | 0.4572 | 618 |
| HOG | Average | 11.5 | 0.9578 | 0.4614 | 418 |
| HOG | Average | 12.0 | 0.9585 | 0.4931 | 295 |
| HOG | Average | 12.5 | 0.9577 | 0.5639 | 195 |
| HOG | Average | 12.75 | 0.9598 | 0.6102 | 157 |
| **HOG** | **Average** | **13.0** | **0.9628** | **0.6511** | **136** |
| HOG | Average | 13.25 | 0.9565 | 0.6327 | 110 |
| HOG | Average | 13.5 | 0.9510 | 0.6109 | 91 |
| HOG | Average | 14.0 | 0.8540 | 0.3175 | 50 |

In [10]:
def calculate_clustering_metrics(file_true, file_pred):
    """
    Calculates Rand Index (RI) and Adjusted Rand Index (ARI) by comparing 
    the predicted clustering with the ground truth.
    
    Args:
        file_true (str): Path to the ground truth TXT file.
        file_pred (str): Path to the txt file generated by the program.
        
    Returns:
        tuple: (ri, ari) scores.
    """
    def parse_to_labels(file_path):
        name_to_label = {}
        with open(file_path, 'r', encoding='utf-8') as f:
            for cluster_id, line in enumerate(f):
                filenames = line.strip().split()
                for name in filenames:
                    name_to_label[name] = cluster_id
        return name_to_label

    labels_true_dict = parse_to_labels(file_true)
    labels_pred_dict = parse_to_labels(file_pred)

    common_names = sorted(list(set(labels_true_dict.keys()) & set(labels_pred_dict.keys())))
    
    if not common_names:
        print("Error: No common filenames found between the two files.")
        return 0.0, 0.0

    y_true = [labels_true_dict[name] for name in common_names]
    y_pred = [labels_pred_dict[name] for name in common_names]

    ri = rand_score(y_true, y_pred)
    ari = adjusted_rand_score(y_true, y_pred)
    
    return ri, ari

#ri_score, ari_score = calculate_clustering_metrics(GROUND_TRUTH, OUTPUT_TXT)

#print("-" * 30)
#print(f"### CLUSTERING EVALUATION ###")
#print(f"Rand Index (RI):          {ri_score:.4f}")
#print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
#print("-" * 30)